# Previous Application Feature Engineering

This notebook builds applicant-level features from the cleaned previous-application data. It creates record-level ratios and flags, then aggregates the general application history, plus separate summaries for approved and refused applications.


## Import libraries


In [3]:
from pathlib import Path

import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 150)
print("Libraries imported successfully.")

Libraries imported successfully.


## Set project paths


In [5]:
current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
input_path = project_root / "data" / "interim" / "previous_application_clean.pkl"
application_path = project_root / "data" / "interim" / "application_clean.pkl"
training_ids_path = project_root / "data" / "modeling" / "splits" / "training_ids.csv"
test_ids_path = project_root / "data" / "modeling" / "splits" / "test_ids.csv"
output_path = project_root / "data" / "features" / "previous_application_features.pkl"
audit_folder = project_root / "reports" / "audits"
output_path.parent.mkdir(parents=True, exist_ok=True)
audit_folder.mkdir(parents=True, exist_ok=True)
for required_path in [input_path, application_path, training_ids_path, test_ids_path]:
    assert required_path.exists(), f"Required file was not found: {required_path}"
print("Clean input:", input_path)
print("Feature output:", output_path)

Clean input: /Users/taranveersingh/A-MRP/data/interim/previous_application_clean.pkl
Feature output: /Users/taranveersingh/A-MRP/data/features/previous_application_features.pkl


## Load cleaned data and split information


In [7]:
previous = pd.read_pickle(input_path)
application_target = pd.read_pickle(application_path)[["SK_ID_CURR", "TARGET"]]
training_ids = pd.read_csv(training_ids_path)["SK_ID_CURR"]
test_ids = pd.read_csv(test_ids_path)["SK_ID_CURR"]
training_id_set = set(training_ids)
test_id_set = set(test_ids)
print("Clean previous-application shape:", previous.shape)
print("Applicants represented:", previous["SK_ID_CURR"].nunique())
print("Contract status counts:")
print(previous["NAME_CONTRACT_STATUS"].value_counts())

Clean previous-application shape: (1413701, 36)
Applicants represented: 291057
Contract status counts:
NAME_CONTRACT_STATUS
Approved        886099
Canceled        259441
Refused         245390
Unused offer     22771
Name: count, dtype: int64


Most previous applications were approved (about 63%), with cancelled and refused applications making up most of the rest.


## Create record-level financial and status features


In [10]:
def safe_ratio(numerator, denominator):
    return (numerator / denominator.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

previous["PREV_CREDIT_APPLICATION_RATIO"] = safe_ratio(previous["AMT_CREDIT"], previous["AMT_APPLICATION"])
previous["PREV_ANNUITY_CREDIT_RATIO"] = safe_ratio(previous["AMT_ANNUITY"], previous["AMT_CREDIT"])
previous["PREV_GOODS_CREDIT_RATIO"] = safe_ratio(previous["AMT_GOODS_PRICE"], previous["AMT_CREDIT"])
previous["PREV_CREDIT_APPLICATION_DIFFERENCE"] = previous["AMT_CREDIT"] - previous["AMT_APPLICATION"]
previous["PREV_ESTIMATED_TOTAL_PAYMENT"] = previous["AMT_ANNUITY"] * previous["CNT_PAYMENT"]
previous["PREV_ESTIMATED_INTEREST"] = previous["PREV_ESTIMATED_TOTAL_PAYMENT"] - previous["AMT_CREDIT"]
previous["PREV_IS_APPROVED"] = previous["NAME_CONTRACT_STATUS"].eq("Approved").astype("int8")
previous["PREV_IS_REFUSED"] = previous["NAME_CONTRACT_STATUS"].eq("Refused").astype("int8")
previous["PREV_IS_CANCELED"] = previous["NAME_CONTRACT_STATUS"].eq("Canceled").astype("int8")
previous["PREV_IS_UNUSED_OFFER"] = previous["NAME_CONTRACT_STATUS"].eq("Unused offer").astype("int8")
previous["PREV_RECENT_APPLICATION"] = previous["DAYS_DECISION"].ge(-365).astype("int8")
previous["PREV_WEEKEND_APPLICATION"] = previous["WEEKDAY_APPR_PROCESS_START"].isin(["SATURDAY", "SUNDAY"]).astype("int8")
print("Record-level features created: 12")

Record-level features created: 12


These describe things like how the credit amount compares with what was requested, an estimate of total interest paid, and flags for each application outcome (approved, refused, cancelled, unused offer).


## Aggregate general previous-application behaviour


In [13]:
previous_features = previous.groupby("SK_ID_CURR").agg(
    PREV_APPLICATION_COUNT=("SK_ID_PREV", "count"),
    PREV_APPROVED_COUNT=("PREV_IS_APPROVED", "sum"),
    PREV_APPROVED_RATE=("PREV_IS_APPROVED", "mean"),
    PREV_REFUSED_COUNT=("PREV_IS_REFUSED", "sum"),
    PREV_REFUSED_RATE=("PREV_IS_REFUSED", "mean"),
    PREV_CANCELED_COUNT=("PREV_IS_CANCELED", "sum"),
    PREV_CANCELED_RATE=("PREV_IS_CANCELED", "mean"),
    PREV_UNUSED_OFFER_COUNT=("PREV_IS_UNUSED_OFFER", "sum"),
    PREV_RECENT_APPLICATION_COUNT=("PREV_RECENT_APPLICATION", "sum"),
    PREV_DAYS_DECISION_MEAN=("DAYS_DECISION", "mean"),
    PREV_DAYS_DECISION_MIN=("DAYS_DECISION", "min"),
    PREV_DAYS_DECISION_MAX=("DAYS_DECISION", "max"),
    PREV_APPLICATION_AMOUNT_MEAN=("AMT_APPLICATION", "mean"),
    PREV_APPLICATION_AMOUNT_MAX=("AMT_APPLICATION", "max"),
    PREV_CREDIT_AMOUNT_MEAN=("AMT_CREDIT", "mean"),
    PREV_CREDIT_AMOUNT_MAX=("AMT_CREDIT", "max"),
    PREV_ANNUITY_MEAN=("AMT_ANNUITY", "mean"),
    PREV_ANNUITY_MAX=("AMT_ANNUITY", "max"),
    PREV_GOODS_PRICE_MEAN=("AMT_GOODS_PRICE", "mean"),
    PREV_PAYMENT_TERM_MEAN=("CNT_PAYMENT", "mean"),
    PREV_PAYMENT_TERM_MAX=("CNT_PAYMENT", "max"),
    PREV_CREDIT_APPLICATION_RATIO_MEAN=("PREV_CREDIT_APPLICATION_RATIO", "mean"),
    PREV_CREDIT_APPLICATION_RATIO_MAX=("PREV_CREDIT_APPLICATION_RATIO", "max"),
    PREV_ANNUITY_CREDIT_RATIO_MEAN=("PREV_ANNUITY_CREDIT_RATIO", "mean"),
    PREV_GOODS_CREDIT_RATIO_MEAN=("PREV_GOODS_CREDIT_RATIO", "mean"),
    PREV_CREDIT_APPLICATION_DIFFERENCE_MEAN=("PREV_CREDIT_APPLICATION_DIFFERENCE", "mean"),
    PREV_ESTIMATED_TOTAL_PAYMENT_MEAN=("PREV_ESTIMATED_TOTAL_PAYMENT", "mean"),
    PREV_ESTIMATED_INTEREST_MEAN=("PREV_ESTIMATED_INTEREST", "mean"),
    PREV_INSURED_APPROVAL_RATE=("NFLAG_INSURED_ON_APPROVAL", "mean"),
    PREV_WEEKEND_APPLICATION_RATE=("PREV_WEEKEND_APPLICATION", "mean"),
    PREV_CREDIT_ABOVE_APPLICATION_RATE=("PREV_CREDIT_ABOVE_APPLICATION", "mean"),
    PREV_ZERO_APPLICATION_RATE=("PREV_ZERO_APPLICATION", "mean"),
    PREV_RECORD_MISSING_RATE_MEAN=("PREV_RECORD_MISSING_RATE", "mean"),
).reset_index()
print("General applicant feature shape:", previous_features.shape)

General applicant feature shape: (291057, 34)


Each applicant's previous applications are summarized here, counts and rates of each outcome, plus average requested and approved amounts.


## Add approved-application summaries


In [16]:
approved = previous.loc[previous["PREV_IS_APPROVED"].eq(1)]
approved_features = approved.groupby("SK_ID_CURR").agg(
    PREV_APPROVED_CREDIT_TOTAL=("AMT_CREDIT", "sum"),
    PREV_APPROVED_CREDIT_MEAN=("AMT_CREDIT", "mean"),
    PREV_APPROVED_CREDIT_MAX=("AMT_CREDIT", "max"),
    PREV_APPROVED_ANNUITY_MEAN=("AMT_ANNUITY", "mean"),
    PREV_APPROVED_TERM_MEAN=("CNT_PAYMENT", "mean"),
    PREV_APPROVED_DAYS_DECISION_MAX=("DAYS_DECISION", "max"),
    PREV_APPROVED_ESTIMATED_INTEREST_MEAN=("PREV_ESTIMATED_INTEREST", "mean"),
).reset_index()
previous_features = previous_features.merge(approved_features, on="SK_ID_CURR", how="left", validate="one_to_one")
print("Applicants with approved history:", len(approved_features))

Applicants with approved history: 290065


Adds a separate summary just for the applications that were actually approved, like total and average approved credit.


## Add refused-application summaries


In [19]:
refused = previous.loc[previous["PREV_IS_REFUSED"].eq(1)]
refused_features = refused.groupby("SK_ID_CURR").agg(
    PREV_REFUSED_CREDIT_MEAN=("AMT_CREDIT", "mean"),
    PREV_REFUSED_CREDIT_MAX=("AMT_CREDIT", "max"),
    PREV_REFUSED_DAYS_DECISION_MAX=("DAYS_DECISION", "max"),
    PREV_REFUSED_APPLICATION_AMOUNT_MEAN=("AMT_APPLICATION", "mean"),
).reset_index()
previous_features = previous_features.merge(refused_features, on="SK_ID_CURR", how="left", validate="one_to_one")
print("Applicants with refused history:", len(refused_features))

Applicants with refused history: 100294


Same idea, but for refused applications. Fewer applicants have refused history (about 100,000 out of 291,000), so these features end up with more missing values.


## Add selected categorical proportions


In [22]:
def clean_feature_name(value):
    return re.sub(r"[^A-Z0-9]+", "_", str(value).upper()).strip("_")

categorical_sources = ["NAME_CONTRACT_TYPE", "NAME_CLIENT_TYPE", "NAME_YIELD_GROUP"]
categorical_feature_count = 0
for column in categorical_sources:
    table = pd.crosstab(previous["SK_ID_CURR"], previous[column], normalize="index")
    prefix = "PREV_" + clean_feature_name(column.replace("NAME_", "")) + "_RATE_"
    table.columns = [prefix + clean_feature_name(c) for c in table.columns]
    table = table.reset_index()
    categorical_feature_count += table.shape[1] - 1
    previous_features = previous_features.merge(table, on="SK_ID_CURR", how="left", validate="one_to_one")
print("Categorical proportion features added:", categorical_feature_count)
print("Applicant feature shape:", previous_features.shape)

Categorical proportion features added: 13
Applicant feature shape: (291057, 58)


Adds the share of each applicant's previous applications that fall into each contract type, client type, and yield group.


## Apply training-only missingness and constant-feature rules


In [25]:
MISSING_THRESHOLD = 0.50
training_base = pd.DataFrame({"SK_ID_CURR": training_ids}).merge(
    application_target, on="SK_ID_CURR", how="left", validate="one_to_one"
).merge(previous_features, on="SK_ID_CURR", how="left", validate="one_to_one")
decision_rows = []
for feature in [c for c in previous_features.columns if c != "SK_ID_CURR"]:
    series = training_base[feature]
    missing_rate = series.isna().mean()
    unique_non_missing = series.nunique(dropna=True)
    correlation = series.corr(training_base["TARGET"]) if unique_non_missing > 1 else np.nan
    decision = "Keep"
    reason = "Retain for global cross-validated feature selection"
    if missing_rate >= MISSING_THRESHOLD:
        decision = "Remove"
        reason = f"Training-applicant missing rate is at least {MISSING_THRESHOLD:.0%}"
    elif unique_non_missing <= 1:
        decision = "Remove"
        reason = "Constant in the training set where values are available"
    decision_rows.append({
        "feature": feature, "missing_count": int(series.isna().sum()),
        "missing_rate": missing_rate, "unique_non_missing": int(unique_non_missing),
        "pearson_target_correlation": correlation,
        "absolute_correlation": abs(correlation) if pd.notna(correlation) else np.nan,
        "decision": decision, "reason": reason,
    })
feature_decisions = pd.DataFrame(decision_rows).sort_values(
    ["decision", "absolute_correlation"], ascending=[True, False]
).reset_index(drop=True)
removed_features = feature_decisions.loc[feature_decisions["decision"] == "Remove", "feature"].tolist()
previous_features = previous_features.drop(columns=removed_features)
print("Features removed:", removed_features)
print("Features retained:", previous_features.shape[1] - 1)
feature_decisions.round(5)

Features removed: ['PREV_REFUSED_DAYS_DECISION_MAX', 'PREV_REFUSED_APPLICATION_AMOUNT_MEAN', 'PREV_REFUSED_CREDIT_MEAN', 'PREV_REFUSED_CREDIT_MAX']
Features retained: 53


,feature,missing_count,missing_rate,unique_non_missing,pearson_target_correlation,absolute_correlation,decision,reason
0,PREV_REFUSED_RATE,13205,0.05368,368,0.07720,0.07720,Keep,Retain for global cross-validated feature sele...
1,PREV_CREDIT_APPLICATION_RATIO_MEAN,14027,0.05702,195863,0.06535,0.06535,Keep,Retain for global cross-validated feature sele...
2,PREV_REFUSED_COUNT,13205,0.05368,45,0.06330,0.06330,Keep,Retain for global cross-validated feature sele...
3,PREV_APPROVED_RATE,13205,0.05368,349,-0.06277,0.06277,Keep,Retain for global cross-validated feature sele...
4,PREV_DAYS_DECISION_MIN,13205,0.05368,2921,0.05529,0.05529,Keep,Retain for global cross-validated feature sele...
5,PREV_CREDIT_APPLICATION_RATIO_MAX,14027,0.05702,86035,0.05072,0.05072,Keep,Retain for global cross-validated feature sele...
6,PREV_GOODS_CREDIT_RATIO_MEAN,14006,0.05693,197824,-0.05026,0.05026,Keep,Retain for global cross-validated feature sele...
7,PREV_DAYS_DECISION_MEAN,13205,0.05368,53707,0.04854,0.04854,Keep,Retain for global cross-validated feature sele...
8,PREV_RECORD_MISSING_RATE_MEAN,13205,0.05368,1373,0.04558,0.04558,Keep,Retain for global cross-validated feature sele...
9,PREV_APPROVED_ANNUITY_MEAN,14022,0.05700,213006,-0.04321,0.04321,Keep,Retain for global cross-validated feature sele...


The four refused-only features were removed here, since most applicants do not have refused history, which pushed their missing rate above 50%. The rest of the refused information (like refused count and rate) stays, since those are filled with 0 for applicants without refused history rather than left missing.


## Validate the applicant-level feature table


In [28]:
numeric_columns = previous_features.select_dtypes(include="number").columns
infinite_count = sum(int(np.isinf(previous_features[c].dropna()).sum()) for c in numeric_columns)
retained_decisions = feature_decisions.loc[feature_decisions["decision"] == "Keep"]
validation_checks = pd.DataFrame([
    {"check": "One row per applicant", "passed": previous_features["SK_ID_CURR"].is_unique},
    {"check": "Only project applicants included", "passed": set(previous_features["SK_ID_CURR"]).issubset(training_id_set.union(test_id_set))},
    {"check": "No TARGET in output", "passed": "TARGET" not in previous_features.columns},
    {"check": "No previous-application ID in output", "passed": "SK_ID_PREV" not in previous_features.columns},
    {"check": "No infinite numerical values", "passed": infinite_count == 0},
    {"check": "No retained feature reaches 50 percent training missingness", "passed": not retained_decisions["missing_rate"].ge(MISSING_THRESHOLD).any()},
    {"check": "Final test excluded from feature decisions", "passed": not training_base["SK_ID_CURR"].isin(test_id_set).any()},
    {"check": "Application counts are positive", "passed": previous_features["PREV_APPLICATION_COUNT"].gt(0).all()},
])
assert validation_checks["passed"].all(), "At least one previous-application feature check failed."
validation_checks

,check,passed
0,One row per applicant,True
1,Only project applicants included,True
2,No TARGET in output,True
3,No previous-application ID in output,True
4,No infinite numerical values,True
5,No retained feature reaches 50 percent trainin...,True
6,Final test excluded from feature decisions,True
7,Application counts are positive,True


All checks passed.


## Save features and audit reports


In [31]:
previous_features.to_pickle(output_path)
feature_decisions.to_csv(audit_folder / "previous_application_engineered_feature_decisions.csv", index=False)
validation_checks.to_csv(audit_folder / "previous_application_feature_engineering_validation.csv", index=False)
print("Previous-application feature table saved:", output_path)
print("Output rows:", len(previous_features))
print("Output columns:", previous_features.shape[1])
print("Applicants represented:", previous_features["SK_ID_CURR"].nunique())
print("Remaining numerical missing values:", int(previous_features.select_dtypes(include="number").isna().sum().sum()))

Previous-application feature table saved: /Users/taranveersingh/A-MRP/data/features/previous_application_features.pkl
Output rows: 291057
Output columns: 54
Applicants represented: 291057
Remaining numerical missing values: 15150


## Main feature engineering results

This notebook built 58 applicant-level features from the cleaned previous-application data, covering general application history plus separate summaries for approved and refused applications.

Four refused-only features were removed for being too sparse, since most applicants do not have refused history. All other checks passed, and the final feature table has 54 columns for the 291,057 applicants with previous-application history. The next step is to build features from the instalment-payment data.
